# 1. 라이브러리 로드

In [2]:
import pandas as pd

from datetime import datetime

from tqdm import tqdm

import sys
sys.path.append('C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset')

import warnings
warnings.filterwarnings('ignore')

# 2. 데이터 로드

## 2.1 아파트 매매 거래량

In [ ]:
# apt_volume_path = 'C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset/apt_volume/apt_purchase_volume_'

# # 초기값 설정
# start_year, start_month = 2010, 6
# end_year, end_month = 2024, 6

# # 1년 단위 파일명 리스트 컴프리헨션 생성
# date_index_lst = [
#     f"{year:04}{start_month:02}_{year+1:04}{start_month-1:02}"
#     for year in range(start_year, end_year)
# ]

# # 마지막 파일 추가 (202406_202412)
# date_index_lst.append("202406_202412")

In [ ]:
# # 행정구 / 월 단위 데이터셋 생성
# monthly_apt_volume_df_lst = []
# for date_index in tqdm(date_index_lst):

#     origin_apt_volume_df = pd.read_excel(f'C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset/apt_volume/apt_purchase_volume_{date_index}.xlsx', header=12)

#     origin_apt_volume_df['region'] = origin_apt_volume_df['시군구'].map(lambda x: ' '.join(x.split(' ')[:2]))
#     origin_apt_volume_df['거래금액(만원)'] = origin_apt_volume_df['거래금액(만원)'].map(lambda x: int(x.replace(',', '')) * 10000)

#     single_monthly_apt_volume_df = origin_apt_volume_df.groupby(['region', '계약년월']).agg(
#                                                                                             volume_cnt=('NO', 'nunique'),
#                                                                                             avg_area_size=('전용면적(㎡)', 'mean'),
#                                                                                             avg_sales=('거래금액(만원)', 'mean'),
#                                                                                             avg_floor_cnt=('층', 'mean')
#                                                                                         ).reset_index().rename(columns={'계약년월' : 'month'})

#     monthly_apt_volume_df_lst.append(single_monthly_apt_volume_df)


# monthly_apt_volume_df = pd.concat(monthly_apt_volume_df_lst, ignore_index=True)

# # data type 변경
# monthly_apt_volume_df['date'] = pd.to_datetime(monthly_apt_volume_df['month'].astype(str), format='%Y%m')

In [ ]:
# monthly_apt_volume_df.to_excel('C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset/apt_volume/apt_purchase_volume_monthly.xlsx')

In [14]:
monthly_apt_volume_df = pd.read_excel('dataset/apt_volume/apt_purchase_volume_monthly.xlsx', index_col=0)
monthly_apt_volume_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4375 entries, 0 to 4374
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   region         4375 non-null   object        
 1   month          4375 non-null   int64         
 2   volume_cnt     4375 non-null   int64         
 3   avg_area_size  4375 non-null   float64       
 4   avg_sales      4375 non-null   float64       
 5   avg_floor_cnt  4375 non-null   float64       
 6   date           4375 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(2), object(1)
memory usage: 273.4+ KB


## 2.2 환율

In [26]:
exchange_rate_df = pd.read_excel('dataset/gov_index/exchange_rate.xlsx')

# 절상율 x, 환율 데이터만 get
exchange_rate_df = exchange_rate_df[~exchange_rate_df['Unnamed: 0'].isnull()].drop('Unnamed: 1', axis=1).T.iloc[1:, [0, 2]]
exchange_rate_df.columns = ['us_exchange_rate', 'jp_exchange_rate']

# date type 변환
exchange_rate_df = exchange_rate_df.reset_index().rename(columns={'index':'date'})
exchange_rate_df['date'] = pd.to_datetime(exchange_rate_df['date'].map(lambda x: str(x[:6])), format='%Y%m')

exchange_rate_df

,date,us_exchange_rate,jp_exchange_rate
0,2010-06-01,"1,222.2","1,380.6"
1,2010-07-01,"1,182.7","1,368.7"
2,2010-08-01,"1,198.1","1,423.8"
3,2010-09-01,"1,140.2","1,368.1"
4,2010-10-01,"1,125.3","1,394.9"
...,...,...,...
170,2024-08-01,"1,336.0",921.2
171,2024-09-01,"1,307.8",922.4
172,2024-10-01,"1,379.9",903.6
173,2024-11-01,"1,394.7",929.4


In [38]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(exchange_rate_df, how='left', on='date')

## 2.3 시장 금리

In [39]:
interest_rate_df = pd.read_excel('dataset/gov_index/interest_rate.xlsx').T

interest_rate_df.columns = interest_rate_df.iloc[0, :]
interest_rate_df = interest_rate_df.iloc[1:]
interest_rate_df = interest_rate_df.reset_index().rename(columns={'index':'date'})

interest_rate_df['date'] = pd.to_datetime(interest_rate_df['date'].map(lambda x: x[:6]), format='%Y%m')

interest_rate_df

Unnamed: 0,date,국고채 3년(평균),국고채 5년(평균),국고채 10년(평균),"회사채 3년(AA-, 평균)",CD 91물(평균),"콜금리(1일물,평균)",기준금리
0,2010-06-01,3.75,4.41,4.93,4.65,2.45,2.0,2.0
1,2010-07-01,3.88,4.45,4.91,4.81,2.58,2.21,2.25
2,2010-08-01,3.73,4.27,4.68,4.68,2.63,2.27,2.25
3,2010-09-01,3.48,3.91,4.28,4.41,2.66,2.27,2.25
4,2010-10-01,3.24,3.66,4.11,4.13,2.66,2.26,2.25
...,...,...,...,...,...,...,...,...
170,2024-08-01,2.92,2.95,3.0,3.42,3.5,3.53,3.5
171,2024-09-01,2.87,2.92,3.01,3.45,3.52,3.53,3.5
172,2024-10-01,2.91,2.97,3.07,3.49,3.43,3.32,3.25
173,2024-11-01,2.86,2.91,3.01,3.43,3.42,3.25,3.0


In [41]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(interest_rate_df, how='left', on='date')

## 2.4 행정구 별 주민등록인구